## 2D convolution



In [1]:
def convolve(signal, kernel):
    n = len(signal)
    k = len(kernel)

    padding = k//2
    signal = [0] * padding + signal + [0]*padding
    output = []

    for i in range(n):
        window = signal[i:i+k]
        val = sum(a*b for a, b in zip(window, kernel))
        output.append(val)
    return output

In [2]:
signal = [1, 2, 3, 4, 5, 6]
kernel = [1, 0, -1]
output = convolve(signal, kernel)
print(output)

[-2, -2, -2, -2, -2, 5]


## 3D convolution

In [12]:
import numpy as np

def convolution(image, kernel):
    """
    image:  (H, W, C_in)
    kernel: (K_h, K_w, C_in, C_out)
    bias:   (C_out,) or None
    returns: (H_out, W_out, C_out)
    """
    
    H, W, C_in = image.shape
    Kh, Kw, C_out = kernel.shape

    pad_h = Kh//2
    pad_w = Kw//2 

    image = np.pad(
            image,
            pad_width=((pad_h, pad_h), (pad_w, pad_w), (0, 0)),
            mode="constant",
            constant_values=0,
        )
    output = np.zeros((H, W, C_out), dtype=np.float32)
    for i in range(H):
        for j in range(W):
            patch = image[i:i+Kh, j:j+Kw, :]
            results = patch[..., None]*kernel
            output[i, j, :] = np.sum(results, axis=(0, 1, 2))
    return output
    

In [13]:
# create an example image and kernel
image = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]], [[9, 10], [11, 12]]])
kernel = np.array([[[1, 0], [0, -1]], [[0, 1], [-1, 0]]])

# perform the convolution operation
output = convolution(image, kernel)

print('Input image:')
print(image)

print('\nKernel:')
print(kernel)

print('\nOutput:')

print(output)

Input image:
[[[ 1  2]
  [ 3  4]]

 [[ 5  6]
  [ 7  8]]

 [[ 9 10]
  [11 12]]]

Kernel:
[[[ 1  0]
  [ 0 -1]]

 [[ 0  1]
  [-1  0]]]

Output:
[[[ -2.   1.]
  [ -3.   1.]]

 [[ -8.   6.]
  [ -6.   2.]]

 [[-16.  14.]
  [ -6.   2.]]]


## Pooling

In [15]:
import numpy as np

def pooling(image, pool_size=2, stride=2, mode="max"):
    """
    image: (H, W, C)
    pool_size: size of pooling window
    stride: stride of pooling
    mode: "max" or "avg"

    returns: pooled output (H_out, W_out, C)
    """

    H, W, C = image.shape

    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1

    output = np.zeros((H_out, W_out, C))

    for i in range(H_out):
        for j in range(W_out):
            patch = image[
                i*stride:i*stride+pool_size,
                j*stride:j*stride+pool_size,
                :
            ]

            if mode == "max":
                output[i, j, :] = np.max(patch, axis=(0,1))
            elif mode == "avg":
                output[i, j, :] = np.mean(patch, axis=(0,1))

    return output

## CNN

In [ ]:
import numpy as np

def relu(x): 
    return np.maximum(0, x)

def conv2d(x, w, b=None, stride=1, padding="same"):
    # x: (H, W, C_in), w: (Kh, Kw, C_in, C_out), b: (C_out,)
    H, W, C_in = x.shape
    Kh, Kw, C_in2, C_out = w.shape
    assert C_in == C_in2

    if b is None:
        b = np.zeros((C_out,), dtype=np.float32)

    if padding == "same":
        pad_h, pad_w = Kh // 2, Kw // 2
    elif padding == "valid":
        pad_h = pad_w = 0
    else:
        pad_h = pad_w = int(padding)

    xpad = np.pad(x, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode="constant")
    Hpad, Wpad, _ = xpad.shape
    H_out = (Hpad - Kh) // stride + 1
    W_out = (Wpad - Kw) // stride + 1

    out = np.zeros((H_out, W_out, C_out), dtype=np.float32)
    for i in range(H_out):
        for j in range(W_out):
            patch = xpad[i*stride:i*stride+Kh, j*stride:j*stride+Kw, :]          # (Kh, Kw, C_in)
            out[i, j, :] = np.sum(patch[..., None] * w, axis=(0, 1, 2)) + b     # (C_out,)
    return out

def maxpool2d(x, pool=2, stride=2):
    # x: (H, W, C)
    H, W, C = x.shape
    H_out = (H - pool) // stride + 1
    W_out = (W - pool) // stride + 1
    out = np.zeros((H_out, W_out, C), dtype=x.dtype)

    for i in range(H_out):
        for j in range(W_out):
            patch = x[i*stride:i*stride+pool, j*stride:j*stride+pool, :]  # (pool, pool, C)
            out[i, j, :] = np.max(patch, axis=(0, 1))
    return out

def cnn_forward(x, params):
    """
    x: (H, W, C_in)
    params: dict with:
      W1: (3,3,C_in,C1), b1: (C1,)
      W2: (3,3,C1,C2),  b2: (C2,)
      W3: (D,C_classes), b3: (C_classes,)
    """
    z1 = conv2d(x, params["W1"], params["b1"], padding="same")
    a1 = relu(z1)
    p1 = maxpool2d(a1, pool=2, stride=2)

    z2 = conv2d(p1, params["W2"], params["b2"], padding="same")
    a2 = relu(z2)
    p2 = maxpool2d(a2, pool=2, stride=2)

    flat = p2.reshape(-1)
    logits = flat @ params["W3"] + params["b3"]
    return logits

# Example usage (random weights)
H, W, C_in = 32, 32, 3
C1, C2, C_classes = 8, 16, 10
x = np.random.randn(H, W, C_in).astype(np.float32)

# after two 2x2 pools: 32->16->8, so feature dim = 8*8*C2
D = 8 * 8 * C2
params = {
    "W1": (0.01 * np.random.randn(3, 3, C_in, C1)).astype(np.float32),
    "b1": np.zeros((C1,), dtype=np.float32),
    "W2": (0.01 * np.random.randn(3, 3, C1, C2)).astype(np.float32),
    "b2": np.zeros((C2,), dtype=np.float32),
    "W3": (0.01 * np.random.randn(D, C_classes)).astype(np.float32),
    "b3": np.zeros((C_classes,), dtype=np.float32),
}
logits = cnn_forward(x, params)
print(logits.shape)  # (10,)